Part A — Environment setup

In [1]:
%pip install -q -U transformers accelerate "gradio"

print("Akhilon S - 24BAD008")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.2 MB/s eta 0:00:00
Akhilon S - 24BAD008


In [2]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch Version:", torch.__version__)
print("Device:", DEVICE)

PyTorch Version: 2.11.0+cu130
Device: cuda


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if DEVICE == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16
    ).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32
    ).to(DEVICE)

model.eval()

print("Model loaded successfully!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully!


Part B — Prompt handling and response generation

In [4]:
print("Akhilon S - 24BAD008")

SYSTEM_MESSAGE = (
    "You are a helpful AI assistant. "
    "Answer clearly and concisely."
)

def generate_response(user_message, history=None):

    if not isinstance(user_message, str) or not user_message.strip():
        return "Please enter a message."

    messages = [{"role": "system", "content": SYSTEM_MESSAGE}]

    # Include previous conversation messages
    for item in list(history or [])[-8:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if role in ["user", "assistant"] and isinstance(content, str):
                if content.strip():
                    messages.append({
                        "role": role,
                        "content": content.strip()
                    })

        elif isinstance(item, (tuple, list)) and len(item) == 2:
            old_user, old_assistant = item

            if old_user:
                messages.append({
                    "role": "user",
                    "content": str(old_user)
                })

            if old_assistant:
                messages.append({
                    "role": "assistant",
                    "content": str(old_assistant)
                })

    messages.append({
        "role": "user",
        "content": user_message.strip()
    })

    try:
        # Format messages for the instruction-following model
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(DEVICE)

        # Generate the answer
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=160,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens
        new_tokens = output_ids[
            0, inputs["input_ids"].shape[1]:
        ]

        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        return response or "Please try another prompt."

    except Exception as error:
        print("Error:", error)
        return "Sorry, an error occurred while generating the response."

Akhilon S - 24BAD008


In [5]:
test_prompts = [
    "Explain artificial intelligence in three simple sentences.",
    "What is a Transformer model in deep learning?",
    "Write a Python function to add two numbers.",
    "Give three tips to save electricity at home.",
    "Write a polite email requesting an assignment deadline extension."
]

for i, prompt in enumerate(test_prompts, start=1):
    print(f"\nTEST {i}")
    print("Prompt:", prompt)

    response = generate_response(prompt, history=[])

    print("Response:", response)
    print("-" * 60)


TEST 1
Prompt: Explain artificial intelligence in three simple sentences.
Response: Artificial intelligence is the ability of computers to perform tasks that typically require human intelligence, such as learning from experience, recognizing patterns, and making decisions without being explicitly programmed. It involves algorithms and large datasets that enable machines to analyze situations, identify rules, and adapt over time, much like humans do.
------------------------------------------------------------

TEST 2
Prompt: What is a Transformer model in deep learning?
Response: A Transformer model in deep learning is an advanced type of neural network that can learn long-range dependencies between words or symbols, allowing it to process text much more efficiently than traditional models like BERT (Bidirectional Encoder Representations from Transformers). It achieves this by using self-attention mechanisms and recurrent connections for sequential data processing. This enables the mo

Part C — Build the chatbot interface

In [6]:
print("Akhilon S - 24BAD008")

demo = gr.ChatInterface(
    fn=generate_response,
    title="Mini LLM Chatbot",
    description=(
        "A simple AI chatbot powered by a pretrained "
        "Transformer language model."
    ),
    examples=[
        "What is artificial intelligence?",
        "Explain machine learning.",
        "Write a Python program to add two numbers.",
        "Give me three study tips."
    ],
    chatbot=gr.Chatbot(height=420)
)

print("Chatbot interface created successfully!")

Akhilon S - 24BAD008
Chatbot interface created successfully!


In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8fe8e888bdebc1d988.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
